# Predict Cache Demo: Run Multiple Metrics Without Re-Running Inference

This notebook shows how the **predict cache** lets you compute multiple metrics on the same model-dataset pair without repeating expensive model inference.

## What You'll Learn

1. **The Problem**: Running different metrics (e.g., global mAP vs. per-class mAP) normally requires running model inference each time
2. **The Solution**: The predict cache saves inference results, so subsequent metric calculations are nearly instant
3. **Why It Matters**: You can use simple, focused metrics without worrying about performance penalties

## What We'll Do

1. Download the VisDrone-2019 test dataset (~279 MB)
2. Run a global mAP metric (this triggers model inference)
3. Run a per-class mAP metric (this uses cached predictions — much faster!)
4. Compare the timing to see the cache in action

## Step 1: Download the Dataset

In [ ]:
import urllib.request
import zipfile
from pathlib import Path
import re
import os

extract_dir = Path("visdrone_test")
# Only pull data if it doesn't exist
if not os.path.exists(extract_dir):
    # VisDrone2019-DET testset-dev from official GitHub repo
    file_id = "1PFdW_VFSCfZ_sTSZAGjQdifF_Xd5mf0V"
    initial_url = f"https://drive.google.com/uc?export=download&id={file_id}"

    zip_path = Path("visdrone_test.zip")

    # First request to get the warning page
    with urllib.request.urlopen(initial_url) as response:
        html = response.read().decode('utf-8')

    # Check if it's a virus scan warning page and extract download URL
    if 'drive.usercontent.google.com/download' in html:
        # Extract the uuid from the form
        uuid_match = re.search(r'name="uuid" value="([^"]+)"', html)
        uuid = uuid_match.group(1) if uuid_match else None

        # Build the actual download URL
        params = f"id={file_id}&export=download&confirm=t"
        if uuid:
            params += f"&uuid={uuid}"

        download_url = f"https://drive.usercontent.google.com/download?{params}"
        urllib.request.urlretrieve(download_url, zip_path)
    else:
        # Direct download worked
        with open(zip_path, 'wb') as f:
            f.write(html.encode('utf-8'))

    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Done! Files in: {extract_dir}")
else:
    print(f"{extract_dir} found on system.  No files downloaded.")

## Step 2: Set Up Evaluation Components

We need four things:
- **Dataset**: VisDrone object detection test split
- **Model**: A pretrained object detector for VisDrone
- **Two Metrics**: Global mAP and per-class mAP (both measure the same thing, but at different granularities)
- **Capability**: `MaiteEvaluation` to run everything

In [ ]:
from jatic_ri.core.object_detection import MaiteEvaluation
from jatic_ri.core.object_detection.dataset_loaders import VisdroneDetectionDataset
from jatic_ri.core.object_detection.models import VisdroneODModel
from jatic_ri.core.object_detection.metrics import map50_torch_metric_factory, multiclass_map50_torch_metric_factory

# Load dataset and model with explicit IDs (required for cache to work correctly)
dataset = VisdroneDetectionDataset(extract_dir, dataset_id="demo_vd2019test")
model = VisdroneODModel(arch="resnet18", model_id="vd_resnet18")

# Create two metrics that measure the same underlying statistic differently
metric_global = map50_torch_metric_factory() # Returns a single mAP score
metric_classwise = multiclass_map50_torch_metric_factory() # Returns mAP per class

# Create the evaluation capability
maite_eval = MaiteEvaluation()

## Step 3: Configure the Cache

Set up a local cache directory to store prediction results.

In [ ]:
from jatic_ri import cache_path

# Set up a local cache directory for this demo
cache_path(Path("demo_cache"))
md_report = ""

print(f"Cache directory: {cache_path()}")

## Step 4: Run Evaluations and See the Cache in Action

Now we run evaluations with both metrics. Watch the timing:

1. **First run (global metric)**: No cache exists, so inference runs fully. Expect 3-5 minutes without GPU.
2. **Second run (classwise metric)**: Cache hit! Predictions are loaded from disk. Expect ~10-15 seconds.
3. **Subsequent runs**: Both prediction and evaluation caches hit. Expect <1 second.

**Try experimenting:**
- Swap the order of the two cells below
- Delete the `demo_cache` folder and re-run
- Set `use_cache=False` to disable caching

In [ ]:
%%time
# Run 1: Global mAP metric (will run full inference if cache is empty)
run_global = maite_eval.run(datasets=[dataset], models=[model], metrics=[metric_global], use_cache=True)
md_report += run_global.collect_md_report(threshold=0.5)

In [ ]:
%%time
# Run 2: Per-class mAP metric (should be fast due to prediction cache hit!)
run_classwise = maite_eval.run(datasets=[dataset], models=[model], metrics=[metric_classwise], use_cache=True)
md_report += run_classwise.collect_md_report(threshold=0.5)

## Step 5: View the Results

Display the evaluation reports from both runs.

In [ ]:
from jatic_ri.core.report._markdown import create_markdown_output

# Save and display the combined report
report_path = cache_path() / "report"
report_path.mkdir(parents=True, exist_ok=True)

report_filename = 'Sample_Report.md'
report = create_markdown_output(md_report, report_path, report_filename, display = True)

---

## Addendum: Technical Background

This section provides additional context for readers interested in the design rationale behind the predict cache and metric return types.

### Why Narrow Metric Return Types?

Metrics implementations like [TorchMetrics Mean Average Precision](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html) return complex data structures when `metric.compute()` is called (see the `map_dict` return value in that link for an example).

However, the less tightly bound the structure of a Metric's return type is, the more complexity the Capability implementer must handle. The [Reference Implementation Conventions on metrics](https://jatic.pages.jatic.net/reference-implementation/reference-implementation/reference/conventions.html) address this by requiring each Metric to have a `return_key` property. This key must be present in the output dictionary with a value that is safely castable to a float.

In other words: if `my_metric.return_key == "MyScore"`, then `my_metric.compute()` will return a dictionary containing at minimum `{"MyScore": 0.5}` (a scalar value).

### The Classwise Metrics Challenge

One could argue that a Capability could handle a `dict[str, Floatable]` and iterate over all entries. But once we introduce classwise metrics, the complexity grows:

- If the value is a scalar, treat it as a "dataset-wide" metric
- If the value is a 1D array, treat it as classwise metrics
- Each classwise metric row needs an associated global metric
- The data structure must specify which array position maps to which class (the "non-consecutive integer index-to-label problem")

The RI's initial solution for classwise metrics structure ([MR #315](https://gitlab.jatic.net/jatic/reference-implementation/reference-implementation/-/merge_requests/315)) required some workarounds—see that discussion for more context.

### How the Predict Cache Addresses This

A valid objection to narrow metric return types: if splitting functionality across multiple metrics requires multiple runs of model inference, aren't we doubling the cost? For example, if TorchMetrics `MeanAveragePrecision` can compute both classwise and global metrics at once, why write two separate RI/MAITE Metrics that each wrap their own instantiation?

The predict cache resolves this. It saves inference results for each model-dataset pairing (identified by their `model_id` and `dataset_id` properties). When you run `evaluate(D, M, G)` followed by `evaluate(D, M, C)`, the second evaluation is nearly instantaneous—the metric is computed fresh, but inference is skipped because predictions are loaded from cache.

Because all Capabilities use the RI's built-in `predict` and `evaluate` functions (which have caching under the hood), they all benefit from this acceleration.

### What This Demo Demonstrates

This demo doesn't answer "what is the appropriate way to narrow metric return type scope"—that's still an evolving design question. But it does demonstrate the cache's usefulness and refutes the performance objection to narrowly-scoped Metrics.